In [0]:
from pyspark.sql.functions import col

relevant_columns = [col for col in ["EmployeeNumber", "EmployeeName", "Attrition", "JobSatisfaction", "Department", "JobRole", "Age", "MonthlyIncome"] if col in df.columns]

high_risk_df = df.filter((df.Attrition == "No") & (df.JobSatisfaction < 3)).select(*relevant_columns)
high_risk_df = high_risk_df.withColumn("EmployeeNumber", col("EmployeeNumber").cast("long"))

high_risk_df.write.format("delta").mode("append").save("/dbfs/tmp/high_risk_attrition_employees1")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Change LongType to IntegerType to match existing table
dummy_schema = StructType([
    StructField("EmployeeNumber", IntegerType(), True),  # Changed from LongType
    StructField("EmployeeName", StringType(), True),
    StructField("Attrition", StringType(), True),
    StructField("JobSatisfaction", IntegerType(), True),
    StructField("Department", StringType(), True),
    StructField("JobRole", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("MonthlyIncome", IntegerType(), True)
])

dummy_data = [(99999, "Dummy User", "No", 1, "Dummy Dept", "Dummy Role", 30, 0)]
dummy_df = spark.createDataFrame(dummy_data, schema=dummy_schema)

dummy_df.write.format("delta").mode("append").save("dbfs:/tmp/high_risk_attrition_employees1")

print("✓ Data appended successfully!")

In [0]:
# To view your current cluster details in Databricks, use the built-in context object
cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", None)
cluster_name = spark.conf.get("spark.databricks.clusterUsageTags.clusterName", None)
profile = spark.conf.get("spark.databricks.cluster.profile")

print(f"Cluster ID: {cluster_id}")
print(f"Cluster Name: {cluster_name}")
print(f"Cluster Profile: {profile}")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Read existing table to get exact schema
existing_df = spark.read.format("delta").load("dbfs:/tmp/high_risk_attrition_employees1")
existing_schema = existing_df.schema
exist


# Use the EXACT same schema
dummy_data = [(99999, "Dummy User", 0, 1, "Dummy Dept", "Dummy Role", 30)]
dummy_df = spark.createDataFrame(dummy_data, schema=existing_schema)

# Now append
dummy_df.write.format("delta").mode("append").save("dbfs:/tmp/high_risk_attrition_employees1")

print("✓ Data appended successfully!")